In [1]:
import pandas as pd
from src.training.bert_pipeline import TrainingBertPipeline
from src.training.bert_truncate_pipeline import BertTruncatePipeline
import logging
import torch
import os

In [2]:
df = pd.read_csv("data/aes_dataset_5k_clean.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5363 entries, 0 to 5362
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   question          4859 non-null   object 
 1   reference_answer  5363 non-null   object 
 2   answer            5363 non-null   object 
 3   score             5363 non-null   float64
 4   normalized_score  5363 non-null   float64
 5   multibert_length  5363 non-null   int64  
 6   dataset           5363 non-null   object 
 7   dataset_num       5363 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 335.3+ KB


In [3]:
df['dataset'].value_counts()
# sag [10]
# sci [10]
# cunlp [10][13]
# stita [10]
# analisis_essay [6]

dataset
sag               2558
analisis_essay    2162
stita              333
cunlp              171
sci                139
Name: count, dtype: int64

In [4]:
# Check if the first file exists
df_result = None
if os.path.exists("experiments/results/results.csv"):
    df_result = pd.read_csv("experiments/results/results.csv")
    print(df_result['config_id'].iloc[-1])
else:
    print("File 'results.csv' does not exist.")

95


In [5]:
batch_sizes = [4, 8]
epochs_list = [5, 10]
learning_rates = [1e-5, 2e-5, 5e-5]
idx = (df_result['config_id'].iloc[-1] + 1) if df_result is not None and not df_result.empty else 0  # index untuk setiap kombinasi
ROOT_DIR = os.getcwd()

In [6]:
# model = [
#     ("bert_length", "bert-base-uncased"),
#     ("indobert_length", "indobenchmark/indobert-base-p1"),
#     ("albert_length", "albert-base-v1"),
#     ("indoalbert_length", "indobenchmark/indobert-lite-base-p2"),
#     ("longformer_length", "allenai/longformer-base-4096"),
#     ("multibert_length", "google-bert/bert-base-multilingual-uncased")
# ]

In [7]:
for batch_size in batch_sizes:
    for num_epochs in epochs_list:
        for lr in learning_rates:
            results = []
            results_epoch = []
            df_result1 = None
            # Check if the second file exists
            if os.path.exists("experiments/results/results_epoch.csv"):
                df_result1 = pd.read_csv("experiments/results/results_epoch.csv")
                print(max(df_result1['valid_qwk']))
            else:
                print("File 'results_epoch.csv' does not exist.")
            config = {
                "df": df,
                "model_name": "google-bert/bert-base-multilingual-uncased",
                "batch_size": batch_size,
                "learning_rate": lr,
                "epochs": num_epochs,
                "config_id": idx,
                "max_seq_len": 128,
                "col_length": "multibert_length",
                "best_valid_qwk": max(df_result1['valid_qwk']) if df_result1 is not None and not df_result1.empty else float("-inf")
            }

            logging.info(
                f"Running configuration: config_id={idx}, model_name={config['model_name']}, batch_size={batch_size}, "
                f"max_seq_length={config['max_seq_len']}, epochs={num_epochs}, learning_rate={lr}"
            )
            
            print(
                f"\nRunning configuration: config_id={idx}, model_name={config['model_name']}, batch_size={batch_size}, "
                f"max_seq_length={config['max_seq_len']}, epochs={num_epochs}, learning_rate={lr}"
            )
            
            try:
                pipeline = BertTruncatePipeline(config, results, results_epoch)
                pipeline.run_training()

                # Save results
                results_path = os.path.join(ROOT_DIR, "experiments/results/results.csv")
                results_epoch_path = os.path.join(ROOT_DIR, "experiments/results/results_epoch.csv")
                TrainingBertPipeline.save_csv(results, results_path)
                TrainingBertPipeline.save_csv(results_epoch, results_epoch_path)
            except Exception as e:
                logging.error(f"Error in config_id={idx}: {str(e)}")
                print(f"Error in config_id={idx}: {str(e)}")
                torch.cuda.empty_cache()
            finally:
                # Clear GPU memory after every configuration
                del pipeline.model
                del pipeline.tokenizer
                del pipeline.optimizer
                torch.cuda.empty_cache()

            idx += 1

0.9251343231432688

Running configuration: config_id=96, model_name=google-bert/bert-base-multilingual-uncased, batch_size=4, max_seq_length=128, epochs=5, learning_rate=1e-05
split dataset run...


Token indices sequence length is longer than the specified maximum sequence length for this model (1001 > 512). Running this sequence through the model will result in indexing errors


create dataset run...
create dataloader run...
create dataloader done...
====== Training Epoch 1/5 ======


c:\Users\User\Documents\Code\env\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Train Loss: 0.0503, Train QWK: 0.6286, Train Pearson: 0.7443
Validation Loss: 0.0350, Validation QWK: 0.8569, Validation Pearson: 0.8846
====== Training Epoch 2/5 ======
Train Loss: 0.0306, Train QWK: 0.7190, Train Pearson: 0.8489
Validation Loss: 0.0210, Validation QWK: 0.9019, Validation Pearson: 0.9049
====== Training Epoch 3/5 ======
Train Loss: 0.0220, Train QWK: 0.7535, Train Pearson: 0.8932
Validation Loss: 0.0210, Validation QWK: 0.9014, Validation Pearson: 0.9033
====== Training Epoch 4/5 ======
Train Loss: 0.0158, Train QWK: 0.7965, Train Pearson: 0.9236
Validation Loss: 0.0226, Validation QWK: 0.8963, Validation Pearson: 0.9039
====== Training Epoch 5/5 ======
Train Loss: 0.0119, Train QWK: 0.8188, Train Pearson: 0.9429
Validation Loss: 0.0248, Validation QWK: 0.8992, Validation Pearson: 0.9037
Test Loss: 0.0214, Test QWK: 0.9137, Test Pearson: 0.9191
0.9251343231432688

Running configuration: config_id=97, model_name=google-bert/bert-base-multilingual-uncased, batch_size=4,

...

split dataset run...
create dataset run...
create dataloader run...
create dataloader done...
====== Training Epoch 1/10 ======
Train Loss: 0.0466, Train QWK: 0.6316, Train Pearson: 0.7587
Validation Loss: 0.0374, Validation QWK: 0.8567, Validation Pearson: 0.8780
====== Training Epoch 2/10 ======
Train Loss: 0.0264, Train QWK: 0.7370, Train Pearson: 0.8690
Validation Loss: 0.0234, Validation QWK: 0.8922, Validation Pearson: 0.8957
====== Training Epoch 3/10 ======
Train Loss: 0.0199, Train QWK: 0.7635, Train Pearson: 0.9030
Validation Loss: 0.0217, Validation QWK: 0.8919, Validation Pearson: 0.8993
====== Training Epoch 4/10 ======
Train Loss: 0.0142, Train QWK: 0.8111, Train Pearson: 0.9318
Validation Loss: 0.0239, Validation QWK: 0.8816, Validation Pearson: 0.8884
====== Training Epoch 5/10 ======
Train Loss: 0.0111, Train QWK: 0.8271, Train Pearson: 0.9470
Validation Loss: 0.0246, Validation QWK: 0.8870, Validation Pearson: 0.8945
====== Training Epoch 6/10 ======
Train Loss: 0.008

Token indices sequence length is longer than the specified maximum sequence length for this model (1001 > 512). Running this sequence through the model will result in indexing errors


split dataset run...
create dataset run...
create dataloader run...
create dataloader done...
====== Training Epoch 1/10 ======
Train Loss: 0.0417, Train QWK: 0.6661, Train Pearson: 0.7850
Validation Loss: 0.0427, Validation QWK: 0.8389, Validation Pearson: 0.8862
====== Training Epoch 2/10 ======
Train Loss: 0.0264, Train QWK: 0.7355, Train Pearson: 0.8701
Validation Loss: 0.0277, Validation QWK: 0.8890, Validation Pearson: 0.9043
====== Training Epoch 3/10 ======
Train Loss: 0.0209, Train QWK: 0.7771, Train Pearson: 0.8986
Validation Loss: 0.0224, Validation QWK: 0.8938, Validation Pearson: 0.8963
====== Training Epoch 4/10 ======
Train Loss: 0.0171, Train QWK: 0.7966, Train Pearson: 0.9178
Validation Loss: 0.0271, Validation QWK: 0.8810, Validation Pearson: 0.8935
====== Training Epoch 5/10 ======
Train Loss: 0.0133, Train QWK: 0.8098, Train Pearson: 0.9367
Validation Loss: 0.0288, Validation QWK: 0.8841, Validation Pearson: 0.8937
====== Training Epoch 6/10 ======
Train Loss: 0.010

Token indices sequence length is longer than the specified maximum sequence length for this model (1001 > 512). Running this sequence through the model will result in indexing errors


create dataset run...
create dataloader run...
create dataloader done...
====== Training Epoch 1/10 ======
Train Loss: 0.0475, Train QWK: 0.6467, Train Pearson: 0.7557
Validation Loss: 0.0490, Validation QWK: 0.7911, Validation Pearson: 0.8778
====== Training Epoch 2/10 ======
Train Loss: 0.0287, Train QWK: 0.7272, Train Pearson: 0.8579
Validation Loss: 0.0230, Validation QWK: 0.8950, Validation Pearson: 0.9009
====== Training Epoch 3/10 ======
Train Loss: 0.0244, Train QWK: 0.7463, Train Pearson: 0.8801
Validation Loss: 0.0278, Validation QWK: 0.8777, Validation Pearson: 0.8853
====== Training Epoch 4/10 ======
Train Loss: 0.0209, Train QWK: 0.7718, Train Pearson: 0.8984
Validation Loss: 0.0243, Validation QWK: 0.8838, Validation Pearson: 0.8869
====== Training Epoch 5/10 ======
Train Loss: 0.0162, Train QWK: 0.7971, Train Pearson: 0.9221
Validation Loss: 0.0333, Validation QWK: 0.8557, Validation Pearson: 0.8870
====== Training Epoch 6/10 ======
Train Loss: 0.0134, Train QWK: 0.8204,